# PropAI — Deal Underwriting Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/propai/blob/main/notebooks/01_underwriting_demo.ipynb)

This notebook demonstrates PropAI's financial engine directly — no server, no Docker, no API keys required.

We'll underwrite a 24-unit multifamily deal and produce:
- Going-in cap rate, DSCR, cash-on-cash
- Levered/unlevered IRR (Newton-Raphson DCF)
- Equity multiple and NPV
- Full 5-year pro forma
- 5×5 IRR sensitivity table (exit cap × rent growth)
- LP/GP equity waterfall

In [ ]:
# Install (Colab / fresh env)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'httpx', 'pandas', 'matplotlib', '-q'], check=True)
print('Dependencies ready')

In [ ]:
# Clone the repo (Colab) or adjust sys.path for local use
import os, sys

if not os.path.exists('propai'):
    os.system('git clone https://github.com/your-org/propai.git --depth=1 -q')
sys.path.insert(0, os.path.join(os.getcwd(), 'propai', 'backend'))
print('PropAI backend on path')

## 1. Define the Deal

In [ ]:
from engine.financial.models import (
    DealInput, LoanInput, OperatingAssumptions, ExitAssumptions
)

deal = DealInput(
    name='The Austin Arms',
    asset_class='multifamily',
    purchase_price=4_800_000,
    units=24,
    market='Austin, TX',
    closing_costs=0.01,
    immediate_capex=0,
    loan=LoanInput(
        ltv=0.70,
        interest_rate=0.0675,
        amortization_years=30,
        loan_type='fixed',
        origination_fee=0.01,
    ),
    operations=OperatingAssumptions(
        gross_scheduled_income=576_000,
        vacancy_rate=0.05,
        credit_loss_rate=0.01,
        other_income=14_400,
        property_taxes=72_000,
        insurance=18_000,
        management_fee_pct=0.05,
        maintenance_reserves=36_000,
        capex_reserves=24_000,
        utilities=12_000,
        other_expenses=8_400,
        rent_growth_rate=0.03,
        expense_growth_rate=0.02,
    ),
    exit=ExitAssumptions(
        hold_period_years=5,
        exit_cap_rate=0.055,
        selling_costs_pct=0.03,
        discount_rate=0.08,
    ),
)

print(f'Deal: {deal.name}')
print(f'Purchase price:   ${deal.purchase_price:,.0f}')
print(f'Equity required:  ${deal.equity_required:,.0f}')
print(f'Loan amount:      ${deal.loan_amount:,.0f}')

## 2. Compute Key Metrics

In [ ]:
from engine.financial import metrics as m

ops = deal.operations
egi = m.effective_gross_income(ops.gross_scheduled_income, ops.vacancy_rate, ops.credit_loss_rate, ops.other_income)
noi = m.net_operating_income(egi, deal)
cap = m.cap_rate(noi, deal.purchase_price)
grm = m.gross_rent_multiplier(deal.purchase_price, ops.gross_scheduled_income)
ds  = m.annual_debt_service(deal)
dscr = m.debt_service_coverage_ratio(noi, ds)
btcf = m.before_tax_cash_flow(noi, ds)
coc  = m.cash_on_cash_return(btcf, deal.equity_required)

print('=' * 50)
print(f'  EGI:               ${egi:>12,.0f}')
print(f'  NOI:               ${noi:>12,.0f}')
print(f'  Cap Rate:          {cap:>12.2%}')
print(f'  GRM:               {grm:>12.2f}x')
print(f'  Annual Debt Svc:   ${ds:>12,.0f}')
print(f'  DSCR:              {dscr:>12.2f}x')
print(f'  Before-Tax CF:     ${btcf:>12,.0f}')
print(f'  Cash-on-Cash:      {coc:>12.2%}')
print('=' * 50)

## 3. IRR & DCF Returns

In [ ]:
from engine.financial.dcf import DCFEngine
from engine.financial.proforma import ProFormaEngine

# Build the full pro forma first (needed for DCF)
pf_engine = ProFormaEngine(deal)
result = pf_engine.underwrite()

print('\nDCF Returns')
print('=' * 50)
print(f'  Levered IRR:       {result.metrics.levered_irr:>12.1%}')
print(f'  Unlevered IRR:     {result.metrics.unlevered_irr:>12.1%}')
print(f'  Levered NPV:       ${result.metrics.levered_npv:>11,.0f}')
print(f'  Equity Multiple:   {result.metrics.equity_multiple:>12.2f}x')
print(f'  Avg CoC:           {result.metrics.avg_cash_on_cash:>12.2%}')
print(f'  Total Profit:      ${result.metrics.total_profit:>11,.0f}')
print('=' * 50)

## 4. 5-Year Pro Forma

In [ ]:
import pandas as pd

pf_data = [
    {
        'Year': yr.year,
        'EGI': yr.effective_gross_income,
        'OpEx': yr.operating_expenses,
        'NOI': yr.net_operating_income,
        'Debt Svc': yr.debt_service,
        'BTCF': yr.before_tax_cash_flow,
        'CoC': yr.coc_return,
        'Loan Bal': yr.loan_balance,
    }
    for yr in result.pro_forma
]

df = pd.DataFrame(pf_data).set_index('Year')
pd.options.display.float_format = '{:,.0f}'.format

# Format CoC as percentage
coc_col = df.pop('CoC').map('{:.1%}'.format)
print(df.to_string())
print('\nCash-on-Cash by Year:')
print(coc_col.to_string())

## 5. Cash Flow Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

years = [yr.year for yr in result.pro_forma]
noi_vals  = [yr.net_operating_income for yr in result.pro_forma]
ds_vals   = [yr.debt_service for yr in result.pro_forma]
btcf_vals = [yr.before_tax_cash_flow for yr in result.pro_forma]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#102a43')
ax.set_facecolor('#102a43')

x = range(len(years))
w = 0.3
ax.bar([xi - w/2 for xi in x], noi_vals,  width=w, label='NOI',         color='#10b981', alpha=0.8)
ax.bar([xi + w/2 for xi in x], ds_vals,   width=w, label='Debt Service', color='#334e68', alpha=0.9)
ax.plot(x, btcf_vals, 'o-', color='#e6b800', linewidth=2.5, markersize=6, label='Before-Tax CF', zorder=5)

ax.set_xticks(list(x))
ax.set_xticklabels([f'Year {y}' for y in years], color='#829ab1')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.tick_params(colors='#829ab1')
ax.spines[:].set_color('#243b53')
ax.legend(facecolor='#243b53', labelcolor='#bcccdc', edgecolor='#334e68')
ax.set_title(f'{deal.name} — Cash Flow Projections', color='#d9e2ec', pad=12)
ax.grid(axis='y', color='#243b53', linewidth=0.8)

plt.tight_layout()
plt.show()

## 6. IRR Sensitivity Table

In [ ]:
import pandas as pd

tbl = result.irr_sensitivity

df_sens = pd.DataFrame(
    tbl.data,
    index=[f'Exit {v:.2%}' for v in tbl.row_values],
    columns=[f'Rent {v:.1%}' for v in tbl.col_values],
)

print('\nLevered IRR Sensitivity — Exit Cap Rate × Rent Growth')
print(df_sens.applymap('{:.1%}'.format).to_string())

# Styled HTML version for Jupyter
def color_irr(val):
    if val >= 0.20: return 'background: #064e3b; color: #6ee7b7'
    if val >= 0.15: return 'background: #065f46; color: #a7f3d0'
    if val >= 0.10: return 'background: #78350f; color: #fcd34d'
    if val >= 0.05: return 'background: #7f1d1d; color: #fca5a5'
    return 'background: #450a0a; color: #f87171'

styled = df_sens.style.applymap(color_irr).format('{:.1%}')
styled

## 7. LP/GP Equity Waterfall

In [ ]:
from engine.financial.models import EquityStructure, WaterfallTier
from engine.financial.waterfall import WaterfallEngine

equity_structure = EquityStructure(
    lp_equity_pct=0.90,
    gp_equity_pct=0.10,
    preferred_return=0.08,
    tiers=[
        WaterfallTier(lp_irr_hurdle=0.12, gp_promote=0.20),
        WaterfallTier(lp_irr_hurdle=0.18, gp_promote=0.30),
    ]
)

wf_engine = WaterfallEngine(deal, equity_structure)
wf = wf_engine.compute(result)

print('\nEquity Waterfall Distribution')
print('=' * 55)
for tier in wf.tiers:
    print(f'  {tier.tier_name:<30} ${tier.distributions:>10,.0f}')
    print(f'    LP: ${tier.lp_share:>10,.0f}   GP: ${tier.gp_share:>10,.0f}')
print('-' * 55)
print(f'  Total Distributions:           ${wf.total_distributions:>10,.0f}')
print(f'  LP Total:                      ${wf.lp_total:>10,.0f}  ({wf.lp_total/wf.total_distributions:.1%})')
print(f'  GP Total:                      ${wf.gp_total:>10,.0f}  ({wf.gp_total/wf.total_distributions:.1%})')
print(f'  LP IRR:                        {wf.lp_irr:>12.1%}')
print(f'  GP IRR:                        {wf.gp_irr:>12.1%}')
print('=' * 55)

---

## Next Steps

- **[Notebook 2](./02_market_analysis.ipynb)** — Pull live market data (Census, FRED, HUD, Zillow) for any US metro
- **[Notebook 3](./03_ai_memo_generation.ipynb)** — Generate an institutional investment memo with Claude
- **[GitHub](https://github.com/your-org/propai)** — Star the repo, contribute, or run the full stack with `docker-compose up`